In [22]:
# Import the required packages
import os
from os.path import dirname as one_folder_up, join as join_paths
import sqlite3

# Import pandas to read query outputs as dfs, read source files, numpy 
import pandas as pd
import numpy as np

from datetime import datetime

# For printing exceptions and logging
import traceback
import logging

In [23]:
# Define the functions

# START - Utils
# For separating log sections
def log_divider():
    return "-" * 100
# END - Utils

# START - DB
# Connects to an sqlite db and returns the conn object
def connect_to_database(db_path):
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except Exception as e:
        return False

# Execute a select query against a DB
def execute_db_select(conn, query):
    logging.info(log_divider())
    try:
        logging.info(f"Running the SELECT query: {query}")
        df = pd.read_sql_query(query, conn)
        logging.info(f"SELECT successful! {len(df)} rows returned")
        return df
    except Exception as e:
        logging.warning(f"SELECT unsuccessful!")
        logging.error(traceback.format_exc())
        print(traceback.format_exc())
        return False

# Gets an insert query along with a pandas dataframe, inserts the dataframe into the DB  
def execute_db_insert(conn, query, data):
    logging.info(log_divider())
    try:
        logging.info(f"INSERTing using the query: {query}")
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, data)
        conn.commit()
        cur.close()
        logging.info(f"INSERT successful! {len(data)} rows inserted")
        return True
    except Exception as e:
        logging.warning(f"INSERT unsuccessful!")
        logging.error(traceback.format_exc())
        return False

# Deletes rows from a table based on an array of ids for a particular column
def execute_db_delete(conn, query, ids):
    logging.info(log_divider())
    try:
        logging.info(f"DELETEing using the query: {query}")
        logging.info(f"IDs to delete: {ids}")
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, ids)
        conn.commit()
        cur.close()
        logging.info(f"DELETE successful! {len(ids)} rows deleted")
        return True
    except Exception as e:
        logging.warning(f"DELETE unsuccessful!")
        logging.error(traceback.format_exc())
        return False

def execute_query(conn, query):
    logging.info(log_divider())
    try:
        logging.info(f"EXECUTEing the query: {query}")
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.execute(query)
        conn.commit()
        cur.close()
        logging.info(f"EXECUTE successful!")
        return True
    except Exception as e:
        logging.warning(f"EXECUTE unsuccessful!")
        logging.error(traceback.format_exc())
        return False
# END - DB

# START - PANDAS
# Clean the date column
def parse_date_string(string, accepted_date_formats, convert_to_date_format, value_if_cannot_be_parsed = np.nan):
    # Recursively try to parse the string using one of the accepted formats
    for fmt in accepted_date_formats:
        try:
            course_date = datetime.strptime(string, fmt)
            logging.info(f"{string} - Parsed using the format {fmt}")
            # Convert to the standard format and return
            return datetime.strftime(course_date, convert_to_date_format)
        except:
            pass
            
    logging.warning(f"{string} - Cannot be parsed using any of the accepted formats! Try adding this to the list of accepted formats")
    return value_if_cannot_be_parsed
    
# END - PANDAS

In [24]:
# Define metadata variables
_JOB_START_DATETIME_    = datetime.now()
_JOB_START_DATETIME_STR_    = _JOB_START_DATETIME_.strftime("%Y_%m_%d_%H_%M_%S")
_JOB_NAME_              = "INGEST_CET_ENROLLMENTS"
_JOB_ID_                = f"{_JOB_NAME_}_{_JOB_START_DATETIME_STR_}"
_JOB_SOURCE_            = "CSV file - enrollments"
_JOB_DESTINATION_       = "SQLite DB - courses/enrollments"

In [25]:
# Define parameters
source_file             = r"enrolments.csv"
target_db               = "courses.db"
target_table            = "enrollments"
target_fields           = [
                            "enrollment_id",
                            "participant_id",
                            "participant_name",
                            "course_id",
                            "course_date",
                            "amount",
                            "subsidy",
                            "credits_used"
                        ]

# Define the accepted date formats and the cleaned date format for the course_date field
accepted_date_formats   = [
                            "%Y/%m/%d", # "2024/04/01"
                            "%d-%m-%Y", # "20-05-2024"
                            "%Y-%m-%d", # "2024-04-15"
                            "%d %b %Y", # "01 May 2024"
                            "%d/%m/%Y", # "22/04/2024"
                        ]
convert_to_date_format  = "%Y-%m-%d"

In [26]:
# Derive required variable(s)
current_folder          = os.getcwd()
root_folder             = one_folder_up(current_folder) 
""" 
    As the current file sits here "\gic_assignment\1_Data_Ingestion_Pipeline\"
    and the root folder is gic_assignment. 
"""

data_folder             = join_paths(root_folder, "data")
input_folder            = join_paths(data_folder, "input")
output_folder           = join_paths(data_folder, "output")

source_file_path        = join_paths(input_folder, source_file)
target_db_path          = join_paths(output_folder, target_db)

# Logging variables
log_folder              = join_paths(current_folder, "logs")
log_file                = f"{_JOB_ID_}.log"
log_file_path           = join_paths(log_folder, log_file)

In [27]:
# Init Logging
# Delete the log file if it already exists
if os.path.exists(log_file_path):
    os.remove(log_file_path)
FORMAT = '%(asctime)s: %(message)s'
logging.basicConfig(filename = log_file_path, format = FORMAT, level = logging.INFO, force = True) # Use force to reset the config

# Log the important info so far
logging.info(f"Job started at {_JOB_START_DATETIME_STR_}")
logging.info(f"Job ID - {_JOB_ID_}, Job Name - {_JOB_NAME_}")
logging.info(f"Job Source - {_JOB_SOURCE_}, Job Destination - {_JOB_DESTINATION_}")
logging.info(log_divider())
logging.info("Context Parameters:")
logging.info(f"Root folder  - {root_folder}")
logging.info(f"Source file path - {source_file_path}")
logging.info(f"Target db path - {target_db_path}")
logging.info(f"Target table - {target_table}")
logging.info(f"Log file path - {log_file_path}")

In [28]:
def main():
    logging.info(log_divider())

    # Create the target db conn
    conn = connect_to_database(target_db_path)
    logging.info(f"Target database connection created  - {conn}")
    
    # Read the input file as a pandas df
    enrollment_df = pd.read_csv(source_file_path)
    logging.info(f"Source file  has been read! {enrollment_df.shape[0]} rows found")

    # Delete rows in the target table if the enrolment IDs are already present
    ids_to_delete = [tuple([id]) for id in list(enrollment_df['enrollment_id'])]
    delete_query = f"DELETE FROM {target_table} WHERE enrollment_id = ?"
    query_status = execute_db_delete(conn, delete_query, ids_to_delete)

    # Check if the delete was successful
    if query_status:
        # Insert rows into the target table
        # Clean the course_date field before inserting (Using apply, lambda and the main parse_date_string function)
        enrollment_df['course_date_cleaned']    = enrollment_df[['course_date']].apply(
                                                    lambda row: parse_date_string(row['course_date'], accepted_date_formats, convert_to_date_format), axis = 1
                                                )

        # Data Quality checks
        # Check 1 - If course_date could not be parsed, flag the record out. Filter these records out from the main df
        course_date_unparsed                    = enrollment_df[pd.isnull(enrollment_df['course_date_cleaned'])]
        course_date_unparsed['Failed_Check']    = "The course date could not be parsed. The format is either not in the list of accepted formats or the date is not valid!"
        enrollment_df                           = enrollment_df[pd.notnull(enrollment_df['course_date_cleaned'])]
        # Check 2 - If course date is too far in the past, flag the record out. Filter these records out from the main df
        # Using the start of SkillsFuture as the threshold
        course_date_too_far_in_past             = enrollment_df[enrollment_df['course_date_cleaned'] < '2015-01-01'] 
        course_date_too_far_in_past['Failed_Check']    = "Date older than SkillsFuture portal!"
        enrollment_df                           = enrollment_df[enrollment_df['course_date_cleaned'] >= '2015-01-01']
        # Combine the records to be flagged out and select only the required columns
        records_to_be_flagged_out               = pd.concat([course_date_unparsed, course_date_too_far_in_past])
        records_to_be_flagged_out               = records_to_be_flagged_out[target_fields + ['Failed_Check']]
        
        # Update the old dataframe with the new cleaned column, drop the old column and rearrange the columns
        enrollment_df                           = enrollment_df[target_fields + ['course_date_cleaned']].drop(
                                                    columns = ['course_date']
                                                ).rename(columns = {'course_date_cleaned': 'course_date'})[target_fields]
        
        # Prepare the data to be inserted
        data = list()
        ids_that_cannot_be_casted = list()
        for index, row in enrollment_df.iterrows():
            try:
                data.append(
                    tuple(
                        [
                            int(row['enrollment_id']),
                            row['participant_id'],
                            row['participant_name'],
                            int(row['course_id']),
                            row['course_date'], # Already handled above
                            float(row['amount']),
                            float(row['subsidy']),
                            float(row['credits_used'])
                        ]
                    )
                )
            except Exception as e:
                logging.warning(f"Some data in the row with id {row['enrollment_id']} could not be cast properly! Quarantining it...")
                ids_that_cannot_be_casted.append(row['enrollment_id'])

        # Update records to be flagged out
        records_that_could_not_be_cast          = enrollment_df[enrollment_df['enrollment_id'].isin(ids_that_cannot_be_casted)]
        records_that_could_not_be_cast['Failed_Check']    = "Some fields do not match the datatype defined in the database table!"
        records_to_be_flagged_out               = pd.concat([records_to_be_flagged_out, records_that_could_not_be_cast])
        records_to_be_flagged_out               = records_to_be_flagged_out[target_fields + ['Failed_Check']]
        
        # Create the insert query using the list of 'target_fields'
        insert_query = f"INSERT INTO {target_table} ({', '.join(target_fields)}) VALUES ({', '.join('?' * len(target_fields))})"
        query_status = execute_db_insert(conn, insert_query, data)

        # Check if the insert was successful
        if query_status:
            logging.info(f"Pipeline complete! Exiting...")
        else:
            # If the insert wasn't successful, close the db connection and exit
            conn.close()
            raise Exception("Encountered an error while inserting the new data! Exiting...")
    else:
        # If the delete wasn't successful, close the db connection and exit
        conn.close()
        raise Exception("Encountered an error while deleting the existing records! Exiting...")

# Call the main function
main()